In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
import copy
import numpy as np
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
%matplotlib inline

# Add the parent directory to the Python path - bad practice, but it's just for the example
import sys
sys.path.append("/home/heydari/FHHI-XAI/")

device = "cuda:0" if torch.cuda.is_available() else "cpu"

from src.glocal_analysis import run_analysis
from datasets.flood_dataset_crp import FloodDataset
from src.datasets.DLR_dataset import DatasetDLR
from src.plot_crp_explanations import plot_explanations, plot_one_image_explanation
from src.minio_client import MinIOClient
from LCRP.models import get_model
from LCRP.utils.crp_configs import ATTRIBUTORS, CANONIZERS, VISUALIZATIONS, COMPOSITES

import logging
# Suppress specific noisy libraries if needed
logging.getLogger("PIL").setLevel(logging.WARNING)
logging.getLogger("matplotlib").setLevel(logging.WARNING)
logging.getLogger("numba").setLevel(logging.WARNING)



In [ ]:
# Define transformation (if needed)
transform = transforms.Compose([
    transforms.ToTensor(),  # Convert to tensor
])
# Load dataset
root_dir = "/home/heydari/FHHI-XAI/data/General_Flood_v3"
dataset = FloodDataset(root_dir=root_dir, split="train", transform=transform)
#device = "cuda:0" if torch.cuda.is_available() else "cpu"
device = "cpu"
model_name = "pidnet"
#device = "cuda:0" if torch.cuda.is_available() else "cpu"
ckpt_path = "/home/heydari/FHHI-XAI/models/flood_model.pt"

# Loading unet with path to checkpoint
model = get_model(model_name=model_name, ckpt_path=ckpt_path, device=device)

model.augment = False
output_dir = "/home/heydari/FHHI-XAI/examples/output/crp/pidnet_flood_old_flood_model"

In [ ]:
print(device)

In [ ]:
output_dir = "/home/heydari/FHHI-XAI/examples/output/crp/pidnet_flood_old_flood_model"

# CRP

In [ ]:
run_analysis(model_name, model, dataset, output_dir=output_dir, device=device)

In [ ]:
# Setting up main parameters
class_id = 1
sample_id = 70
n_concepts = 3
n_refimgs = 12
layer = "layer5.0.conv1"
# layer = "layer1.1.conv1"
mode = "relevance"
prediction_num = 0

# if failing, try to restart the notebook and do not run analysis again, go directly to plotting
plot_explanations(model_name, model, dataset, sample_id, class_id, layer, prediction_num, mode, n_concepts, n_refimgs, output_dir=output_dir)

In [ ]:
# ===== Plot Explanations with 4 Prototypes and 5 Concepts =====

import os
import numpy as np
import matplotlib.pyplot as plt
from src.plot_pcx_proto_concept_matrix import plot_proto_concept_matrix
from crp.helper import get_layer_names

# ===== Setup Parameters =====
# Same as plot_explanations but with prototype/concept visualization
model_name = "pidnet"
sample_id = 70
class_id = 1
layer = "layer5.0.conv1"
prediction_num = 0
mode = "relevance"
n_concepts = 5        # 5 concepts as requested
n_refimgs = 12
num_prototypes = 4    # 4 prototypes as requested
output_dir = "/home/heydari/FHHI-XAI/examples/output/crp/pidnet_flood_old_flood_model"

# ===== Step 1: Run plot_explanations (CRP explanation for single sample) =====
print("="*60)
print("Step 1: Plotting CRP explanation for sample")
print("="*60)
plot_explanations(
    model_name, 
    model, 
    dataset, 
    sample_id, 
    class_id, 
    layer, 
    prediction_num, 
    mode, 
    n_concepts, 
    n_refimgs, 
    output_dir=output_dir
)

# ===== Step 2: Plot prototypes and concepts for the same layer =====
print("\n" + "="*60)
print("Step 2: Plotting Prototypes & Concepts Matrix")
print("="*60)

# Load precomputed attributions for the layer
try:
    layer_name = layer
    attributions_path = f"{output_dir}/{layer_name}/attributions.npy"
    
    if os.path.exists(attributions_path):
        attributions = np.load(attributions_path)
        print(f"Loaded attributions from {attributions_path}")
        
        # Create proto-concept visualization
        fig, meta = plot_proto_concept_matrix(
            model_name=model_name,
            model=model,
            dataset=dataset,
            layer_name=layer_name,
            attributions=attributions,
            num_prototypes=num_prototypes,        # 4 prototypes
            n_concepts=n_concepts,                # 5 concepts
            n_refimgs=n_refimgs,                  # 12 reference images per concept
            samples_per_proto=3,                  # 3 samples per prototype
            class_id=class_id,
            ref_imgs_path=f"{output_dir}/ref_imgs_pidnet/",
            output_dir_crp=output_dir,
            ref_thumb_size=(320, 320),
            ref_col_scale=2.6,
            concept_row_scale=1.7,
        )
        
        print("\n✓ Proto-Concept Matrix plotted successfully!")
        print(f"  - Prototypes: {num_prototypes}")
        print(f"  - Concepts: {n_concepts}")
        print(f"  - Reference images per concept: {n_refimgs}")
        if "concept_ids" in meta:
            print(f"  - Selected concept IDs: {meta['concept_ids']}")
        
        plt.tight_layout()
        plt.show()
        
    else:
        print(f"⚠ Warning: Attributions file not found at {attributions_path}")
        print("Make sure to run the CRP analysis first (cell with run_analysis)")
        
except Exception as e:
    print(f"❌ Error creating proto-concept matrix: {e}")
    import traceback
    traceback.print_exc()

print("\n" + "="*60)
print("Done!")
print("="*60)

In [ ]:
# ==== Why you're seeing: IndexError: list index out of range ====
# Most likely one of these is true:
# 1) self.image_files / self.mask_files ended up EMPTY (0 items), because
#    filenames don’t align by stem (e.g., image: "123.jpg" vs mask: "123_mask.png").
# 2) The requested sample_id >= len(dataset).
# 3) Wrong split/path (e.g., directories exist but contain no matching files).
#
# Run this quick diagnostic to see what’s going on:

from pprint import pprint
import os

def debug_flooddataset_paths(ds):
    print("— FloodDataset debug —")
    print("image_dir:", ds.image_dir, "exists:", os.path.isdir(ds.image_dir))
    print("mask_dir :", ds.mask_dir,  "exists:", os.path.isdir(ds.mask_dir))
    print("#images:", len(ds.image_files))
    print("#masks :", len(ds.mask_files))
    if len(ds.image_files) > 0:
        print("first 5 images:", ds.image_files[:5])
    if len(ds.mask_files) > 0:
        print("first 5 masks :", ds.mask_files[:5])

# Example:
debug_flooddataset_paths(dataset)
print("dataset length:", len(dataset))
sample_id = min(sample_id, len(dataset)-1)
img, m = dataset[sample_id]


In [ ]:
print("— FloodDataset debug —")
print("image_dir:", dataset.image_dir, os.path.isdir(dataset.image_dir))
print("mask_dir :", dataset.mask_dir,  os.path.isdir(dataset.mask_dir))
print("#images:", len(dataset.image_files))
print("#masks :", len(dataset.mask_files))
print("dataset length:", len(dataset))

sample_id = min(sample_id, len(dataset)-1)
img_t, m_t = dataset[sample_id]
print("img:", img_t.shape, img_t.dtype, img_t.min().item(), img_t.max().item())
print("msk:", m_t.shape,  m_t.dtype,  m_t.unique(sorted=True)[:5])


# PCX

In [ ]:
import os
from torch.utils.data import DataLoader
from crp.concepts import ChannelConcept
from crp.helper import get_layer_names
from datetime import datetime
from tqdm import tqdm
from torchvision.utils import make_grid

# === CRP & Zennit ===
import zennit.image as zimage
from crp.image import imgify
from crp.concepts import ChannelConcept
from crp.helper import get_layer_names

In [ ]:
dataloader = DataLoader(dataset, batch_size=4, shuffle=False, num_workers=8)
cc = ChannelConcept()

layer_names = get_layer_names(model, [torch.nn.Conv2d])
# # Setting up CRP 
attribution = ATTRIBUTORS[model_name](model)
composite = COMPOSITES[model_name](canonizers=[CANONIZERS[model_name]()])
condition = [{"y": 1}]
fv = VISUALIZATIONS[model_name](attribution,
                                 dataset,
                                 layer_names,
                                 preprocess_fn=lambda x: x,
                                 path=output_dir,
                                 max_target="max")

 # Runs faster on MPS
device = "cpu"
model.to(device)
model.eval()

start = datetime.now()

activations = {}
attributions = {}
for i, (x, y) in enumerate(tqdm(dataset)):
     x = x.to(device).requires_grad_()
     condition = [{"y": 1}]
     attr = attribution(x.unsqueeze(0), condition, composite, record_layer=layer_names)

     for layer_name in layer_names:
         if layer_name in attr.activations.keys():
             if layer_name not in attributions.keys():
                 attributions[layer_name] = []
                 activations[layer_name] = []
             activations[layer_name].append(attr.activations[layer_name].amax((-2, -1)))
             attributions[layer_name].append(cc.attribute(attr.relevances[layer_name], abs_norm=True))
for layer_name in layer_names:
     #if layer_name in attribution.keys():
         attributions[layer_name] = torch.cat(attributions[layer_name])
         activations[layer_name] = torch.cat(activations[layer_name])
         folder = f"/home/heydari/FHHI-XAI/examples/output/pcx/pidnet_flood_old_flood_model/{layer_name}/"
         # attributions[layer_name] = torch.cat(attributions[layer_name])
         # activations[layer_name] = torch.cat(activations[layer_name])
         os.makedirs(folder, exist_ok=True)
         np.save(folder + "attributions", attributions[layer_name].cpu().numpy())
         np.save(folder + "activations", activations[layer_name].cpu().numpy())
end = datetime.now()

In [ ]:
layer_names = get_layer_names(model, [torch.nn.Conv2d])


In [ ]:
# Loading
layer_names = get_layer_names(model, [torch.nn.Conv2d])
layer_name = layer_names[50]
print(layer_name)
folder = f"/home/heydari/FHHI-XAI/examples/output/pcx/pidnet_flood_old_flood_model/{layer_name}/"
attributions = torch.from_numpy(np.load(folder + "attributions.npy"))
activations = torch.from_numpy(np.load(folder + "activations.npy"))
indices = np.arange(len(dataset))

num_prototypes = 7

In [ ]:
from umap import UMAP


embedding_attr = UMAP(n_neighbors=5, random_state=123, n_jobs=1)
X_attr = embedding_attr.fit_transform(attributions.detach().cpu().numpy())
x_attr, y_attr = X_attr[:, 0], X_attr[:, 1]

embedding_act = UMAP(n_neighbors=5, random_state=123, n_jobs=1)
X_act = embedding_act.fit_transform(activations.detach().cpu().numpy())
x_act, y_act = X_act[:, 0], X_act[:, 1]

In [ ]:
from scipy import stats
from sklearn.mixture import GaussianMixture

N_PROTOTYPES = 8

fig, axes = plt.subplots(1, 2, dpi=300, figsize=(5, 3), facecolor='white')
for i, X in enumerate([X_attr, X_act]):
    x, y = X[:, 0], X[:, 1]
    xmin = x.min() - 2
    xmax = x.max() + 2
    ymin = y.min() - 2
    ymax = y.max() + 2
    X, Y = np.mgrid[xmin:xmax:100j, ymin:ymax:100j]
    positions = np.vstack([X.ravel(), Y.ravel()])
    values = np.vstack([x, y])
    kernel = stats.gaussian_kde(values, 0.5)
    Z = np.reshape(kernel(positions).T, X.shape).T
    axes[i].contour(Z, extent=[xmin, xmax, ymin, ymax], cmap="Greys", alpha=0.3, extend='min', vmax=Z.max() * 1, zorder=0)
    axes[i].scatter(x, y, s=3, alpha=0.7, zorder=1)
    axes[i].set_xticks([])
    axes[i].set_yticks([])
    axes[i].set_title(["attributions", "activations"][i])
    axes[i].plot([1], y[1], 'ko', markersize=5, label="test image")
    axes[i].legend()


prototypes = []
gmms = []
for i, (X, emb) in enumerate([(attributions, embedding_attr), (activations, embedding_act)]):
    gmms.append(GaussianMixture(n_components=N_PROTOTYPES, random_state=0).fit(X.detach().cpu().numpy()))
    prototypes.append(gmms[-1].means_)
    mean = emb.transform(gmms[-1].means_)
    axes[i].scatter(mean[:, 0], mean[:, 1], s=30, c="#1B4365", zorder=2, label="prototypes")
    for k, prot in enumerate(mean):
        axes[i].text(prot[0], prot[1], k, fontsize=4, color="white", ha="center", va="center")
    axes[i].legend()

plt.tight_layout()
fig.show()

In [ ]:
proto_attr = prototypes[0]

distances = np.linalg.norm(attributions[:, None, :].detach().cpu() - proto_attr, axis=2)
prototype_samples = np.argsort(distances, axis=0)[:8]
prototype_samples = indices[prototype_samples]

fig, axs = plt.subplots(1, N_PROTOTYPES, figsize=(1*N_PROTOTYPES, 8), dpi=200, facecolor='white')


for i in range(N_PROTOTYPES):
    grid = make_grid(
        [dataset[prototype_samples[j][i]][0] for j in range(8)],
        nrow=1,
        padding=0)
    grid = np.array(zimage.imgify(grid.detach().cpu()))
    img = imgify(grid)
    axs[i].imshow(img)
    axs[i].set_xticks([])
    axs[i].set_yticks([])
    axs[i].set_title(f"{i}")


In [ ]:
print(layer_name)

In [ ]:
print(model)

In [ ]:
print(dataset)

In [ ]:
#cell 10

import os
from src.plot_pcx_pidnet_new import plot_pcx_explanations



device = "cuda:0" if torch.cuda.is_available() else "cpu"

# Use parent-relative paths because this notebook lives in the `examples/` folder
output_dir_crp = "/home/heydari/FHHI-XAI/examples/output/crp/pidnet_flood_old_flood_model/"
output_dir_pcx = "/home/heydari/FHHI-XAI/examples/output/pcx/pidnet_flood_old_flood_model/"

#p=7 layer_name='layer3.0.conv1',
#100
#560
#556

#p=8 layer_name='layer3.0.conv1',
#100
#560
#sample_id=1045,

#p=9 l=3.0
#1045



plot_pcx_explanations(
    "pidnet",
    model.to(device),
    dataset,
    sample_id=10,
    layer_name='layer4.0.conv1',
    n_concepts=3,
    n_refimgs=12,
    num_prototypes=5,
    ref_imgs_path="/home/heydari/FHHI-XAI/examples/output/ref_img_flood_old_flood_model",
    output_dir_crp=output_dir_crp,
    output_dir_pcx=output_dir_pcx,
)


In [ ]:
print(layer_name)

In [ ]:
x, y = dataset[200]
print("Sample shape:", x.shape)


In [ ]:
dummy = torch.randn(1, 3, 256, 256)  # try with 3 channels first
output = model(dummy)
print("Output shape:", output.shape)


In [ ]:
import numpy as np
from src.plot_pcx_proto_concept_matrix import plot_proto_concept_matrix
from src.plot_pcx_pidnet_new import get_ref_images

# --- required ---
layer_name = "layer4.0.conv1"  # change to your layer
# layer_name = "layer2.1.conv1"  # change to your layer
attributions = np.load(f"/home/heydari/FHHI-XAI/examples/output/pcx/pidnet_flood_old_flood_model/{layer_name}/attributions.npy")

# --- plot ---
fig, meta = plot_proto_concept_matrix(
    model_name="pidnet",
    model=model,
    dataset=dataset,
    layer_name=layer_name,
    attributions=attributions,
    num_prototypes=8,
    n_concepts=5,
    n_refimgs=3,
    samples_per_proto=3,
    class_id=1,
    ref_imgs_path="/home/heydari/FHHI-XAI/examples/output/ref_img_flood_old_flood_model/",
    output_dir_crp="/home/heydari/FHHI-XAI/examples/output/crp/pidnet_flood_old_flood_model",
    #concept_ids=[28,6],  # optional: force exact concept rows
    skip_prototypes=[0,1,2,3],              # optional: hide a prototype column
    ref_thumb_size=(320, 320),     # make concept refs bigger/clearer
    ref_col_scale=2.6,            # widen concept column
    concept_row_scale=1.7,       # taller concept rows
)

fig
# fig.savefig("/home/heydari/FHHI-XAI/output/pcx/proto_concept_matrix.png", dpi=300)

In [ ]:
import os
from crp.helper import load_maximization, get_layer_names
from LCRP.utils.render import vis_opaque_img_border
from crp.concepts import ChannelConcept
#from src.pcx_helper import get_ref_images
import torch
from LCRP.utils.crp_configs import ATTRIBUTORS, CANONIZERS, VISUALIZATIONS, COMPOSITES
# 
# # Setup
#
model_name ="pidnet"
output_dir_crp = "/home/heydari/FHHI-XAI/examples/output/crp/pidnet_flood_old_flood_model"
ref_imgs_path= "/home/heydari/FHHI-XAI/examples/output/ref_img_flood_old_flood_model"
# 
# # ============================================================
# # SPECIFY YOUR LAYERS HERE
# # ============================================================
layers_to_process = [
    "layer4.0.conv1"
 ]
# 
# # Classes to process
class_ids = [1]  # 0=person, 1=car
# 
# # Number of reference images to save
n_refs = [12]  # Will save both 12 and 6 ref images
# 
# # ============================================================
# # Setup CRP components
# # ============================================================
layer_names_all = get_layer_names(model, types=[torch.nn.Conv2d])
attribution = ATTRIBUTORS[model_name](model)
composite = COMPOSITES[model_name](canonizers=[CANONIZERS[model_name]()])
# 
fv = VISUALIZATIONS[model_name](
     attribution, dataset, layer_names_all,
     preprocess_fn=lambda x: x,
     path=output_dir_crp,
     max_target="max"
 )
# 
# # ============================================================
# # Process each layer
# # ============================================================
crp_relmax_path = os.path.join(output_dir_crp, "RelMax_sum_normed")
# 
for layer_name in layers_to_process:
     print(f"\n{'='*60}")
     print(f"Processing layer: {layer_name}")
     print(f"{'='*60}")
     try:
#         # Load maximization data to get concept count
         d_c_sorted, _, _ = load_maximization(crp_relmax_path, layer_name)
         n_concepts = d_c_sorted.shape[1]
         print(f"  Layer has {n_concepts} concepts, {d_c_sorted.shape[0]} samples")
# 
#         # All concept indices for this layer
         all_concepts = list(range(n_concepts))
# 
         for class_id in class_ids:
             for n_ref in n_refs:
                 ref_path = ref_imgs_path
# 
                 print(f"\n  Class {class_id}, n_ref={n_ref}...")
# 
                 try:
                     ref_imgs = get_ref_images(
                         fv, 
                         topk_ind= all_concepts, 
                         layer_name=layer_name,
                         composite=composite,
                         n_ref=n_ref,
                         ref_imgs_save_path=ref_path
                     )
                     print(f"    ✓ Saved {len(ref_imgs)} concepts")
                 except Exception as e:
                     print(f"    ❌ Error: {e}")
# 
     except Exception as e:
         print(f"  ❌ Error loading layer: {e}")
# 
print("\n" + "="*60)
print("DONE!")
print("="*60)

In [ ]:
import numpy as np
from src.plot_pcx_proto_concept_matrix import plot_proto_concept_matrix

# --- required ---
layer_name = "layer4.0.conv1"  # change to your layer
attributions = np.load(f"/home/heydari/FHHI-XAI/output/pcx/pidnet_old/{layer_name}/attributions.npy")

# --- plot ---
fig, meta = plot_proto_concept_matrix(
    model_name="pidnet",
    model=model,
    dataset=dataset,
    layer_name=layer_name,
    attributions=attributions,
    num_prototypes=8,
    n_concepts=5,
    n_refimgs=3,
    samples_per_proto=3,
    class_id=1,
    ref_imgs_path="/home/heydari/FHHI-XAI/output/ref_imgs_pidnet/",
    output_dir_crp="/home/heydari/FHHI-XAI/output/crp/pidnet_old",
    concept_seed_ids=[6, 28],     # always include these
    concept_select_by="seg_area", # pick remaining by coverage
    concept_fill=True,            # fill remaining rows
    ref_thumb_size=(320, 320),    # make concept refs bigger/clearer
    ref_col_scale=2.6,            # widen concept column
    concept_row_scale=1.7,        # taller concept rows
)

fig
# fig.savefig("/home/heydari/FHHI-XAI/output/pcx/proto_concept_matrix.png", dpi=300)


In [ ]:
import numpy as np
from src.plot_pcx_proto_concept_matrix import plot_proto_concept_matrix

# --- required ---
layer_name = "layer1.1.conv1"  # change to your layer
attributions = np.load(f"/home/heydari/FHHI-XAI/output/pcx/pidnet_old/{layer_name}/attributions.npy")

# --- plot ---
fig, meta = plot_proto_concept_matrix(
    model_name="pidnet",
    model=model,
    dataset=dataset,
    layer_name=layer_name,
    attributions=attributions,
    num_prototypes=5,
    n_concepts=5,
    n_refimgs=3,
    samples_per_proto=3,
    class_id=1,
    ref_imgs_path="/home/heydari/FHHI-XAI/output/ref_imgs_pidnet/",
    output_dir_crp="/home/heydari/FHHI-XAI/output/crp/pidnet_old",
    #concept_seed_ids=[6, 28],     # always include these
    concept_select_by="seg_area",
    concept_ids=[6, 10, 23, 21, 12],
    concept_fill=True,            # fill remaining rows
    ref_thumb_size=(320, 320),    # make concept refs bigger/clearer
    ref_col_scale=2.6,            # widen concept column
    concept_row_scale=1.7,        # taller concept rows
)

fig
print(meta["concept_ids"])

# fig.savefig("/home/heydari/FHHI-XAI/output/pcx/proto_concept_matrix.png", dpi=300)


In [ ]:
import numpy as np
from src.plot_pcx_proto_concept_matrix import plot_proto_concept_matrix

# --- required ---
layer_name = "layer1.1.conv1"  # change to your layer
attributions = np.load(f"/home/heydari/FHHI-XAI/output/pcx/pidnet_old/{layer_name}/attributions.npy")


fig, meta = plot_proto_concept_matrix(
    model_name="pidnet",
    model=model,
    dataset=dataset,
    layer_name=layer_name,
    attributions=attributions,
    num_prototypes=5,
    n_concepts=5,
    n_refimgs=3,
    samples_per_proto=3,
    class_id=1,
    ref_imgs_path="/home/heydari/FHHI-XAI/output/ref_imgs_pidnet/",
    output_dir_crp="/home/heydari/FHHI-XAI/output/crp/pidnet_old",
    concept_seed_ids=[6],          # keep if you want to force 6
    concept_select_by="seg_area",
    concept_fill=True,
    ref_select_by="seg_area",      # NEW: pick refs with big masks
    ref_pool_size=80,              # optional: larger pool for better picks
    ref_thumb_size=(320, 320),
    ref_col_scale=2.6,
    concept_row_scale=1.7,
)


In [ ]:
import numpy as np
from src.plot_pcx_proto_concept_matrix import plot_proto_concept_matrix

# --- required ---
layer_name = "layer1.1.conv1"  # change to your layer
attributions = np.load(f"/home/heydari/FHHI-XAI/output/pcx/pidnet_old/{layer_name}/attributions.npy")


fig, meta = plot_proto_concept_matrix(
    model_name="pidnet",
    model=model,
    dataset=dataset,
    layer_name=layer_name,
    attributions=attributions,
    num_prototypes=5,
    n_concepts=5,
    n_refimgs=3,
    samples_per_proto=3,
    class_id=1,
    ref_imgs_path="/home/heydari/FHHI-XAI/output/ref_imgs_pidnet/",
    output_dir_crp="/home/heydari/FHHI-XAI/output/crp/pidnet_old",
    concept_seed_ids=[6],
    concept_select_by="seg_area",
    concept_fill=True,
    ref_select_by="seg_area",
    ref_pool_size=80,
    ref_overlay_masks=True,     # NEW: highlight mask on ref images
    ref_mask_alpha=0.35,        # tweak overlay strength
    ref_thumb_size=(320, 320),
    ref_col_scale=2.6,
    concept_row_scale=1.7,
)


In [ ]:
import numpy as np
from src.plot_pcx_proto_concept_matrix import plot_proto_concept_matrix

# --- required ---
layer_name = "layer1.0.conv1"  # change to your layer
attributions = np.load(f"/home/heydari/FHHI-XAI/output/pcx/pidnet_old/{layer_name}/attributions.npy")

# --- plot ---
fig, meta = plot_proto_concept_matrix(
    model_name="pidnet",
    model=model,
    dataset=dataset,
    layer_name=layer_name,
    attributions=attributions,
    num_prototypes=8,
    n_concepts=5,
    n_refimgs=3,
    samples_per_proto=3,
    class_id=1,
    ref_imgs_path="/home/heydari/FHHI-XAI/output/ref_imgs_pidnet/",
    output_dir_crp="/home/heydari/FHHI-XAI/output/crp/pidnet_old",
    concept_seed_ids=[6, 28],     # always include these
    concept_select_by="seg_area", # pick remaining by coverage
    concept_fill=True,            # fill remaining rows
    ref_thumb_size=(320, 320),    # make concept refs bigger/clearer
    ref_col_scale=2.6,            # widen concept column
    concept_row_scale=1.7,        # taller concept rows
)

fig
# fig.savefig("/home/heydari/FHHI-XAI/output/pcx/proto_concept_matrix.png", dpi=300)


In [ ]:
import numpy as np
from src.plot_pcx_proto_concept_matrix import plot_proto_concept_matrix

layer_name = "layer4.2.conv1"  # change to your layer
attributions = np.load(f"/home/heydari/FHHI-XAI/output/pcx/pidnet_old/{layer_name}/attributions.npy")

fig, meta = plot_proto_concept_matrix(
    model_name="pidnet",
    model=model,
    dataset=dataset,
    layer_name=layer_name,
    attributions=attributions,
    num_prototypes=5,
    n_concepts=5,
    n_refimgs=3,
    samples_per_proto=3,
    class_id=1,
    ref_imgs_path="/home/heydari/FHHI-XAI/output/ref_imgs_pidnet/",
    output_dir_crp="/home/heydari/FHHI-XAI/output/crp/pidnet_old",
    proto_thumb_size=(190, 190),   # make prototypes bigger/clearer
    ref_thumb_size=(320, 320),     # make concept refs bigger/clearer
    overlay_masks=True,            # red mask on prototype RGBs
    mask_alpha=0.35,               # stronger red
    ref_col_scale=2.6,            # widen concept column
    concept_row_scale=1.7,       # taller concept rows
)

fig

In [ ]:
import numpy as np
from src.plot_pcx_proto_concept_matrix import plot_proto_concept_matrix

layer_name = "layer5.0.conv1"
attributions = np.load("/home/heydari/FHHI-XAI/output/pcx/pidnet_flood/layer5.0.conv1/attributions.npy")

fig, meta = plot_proto_concept_matrix(
    model_name="pidnet",
    model=model,
    dataset=dataset,
    layer_name=layer_name,
    attributions=attributions,
    num_prototypes=5,
    n_concepts=5,
    n_refimgs=3,
    samples_per_proto=6,
    class_id=1,
    concept_ids=[44, 55, 214, 227, 234],
    skip_prototypes=[1],
    ref_imgs_path="/home/heydari/FHHI-XAI/output/ref_imgs_pidnet/",
    output_dir_crp="/home/heydari/FHHI-XAI/output/crp/pidnet_flood/",
    overlay_masks=False,
    ref_thumb_size=(320, 320),     # make concept refs bigger/clearer
    ref_col_scale=2.6,            # widen concept column
    concept_row_scale=1.7,       # taller concept rows
)

fig
# fig.savefig("/home/heydari/FHHI-XAI/output/pcx/pidnet_flood/layer5.0.conv1/panel_refs3_per_proto_with_matrix.png", dpi=300)

In [ ]:
import numpy as np
from src.plot_pcx_proto_concept_matrix import plot_proto_concept_matrix

layer_name = "layer5.0.conv1"
attributions = np.load("/home/heydari/FHHI-XAI/output/pcx/pidnet_flood/layer5.0.conv1/attributions.npy")

fig, meta = plot_proto_concept_matrix(
    model_name="pidnet",
    model=model,
    dataset=dataset,
    layer_name=layer_name,
    attributions=attributions,
    num_prototypes=3,
    n_concepts=5,
    n_refimgs=3,
    samples_per_proto=3,
    class_id=1,
    #concept_ids=[44, 55, 214, 227, 234],
    ref_imgs_path="/home/heydari/FHHI-XAI/output/ref_imgs_pidnet/",
    output_dir_crp="/home/heydari/FHHI-XAI/output/crp/pidnet_flood/",
    ref_thumb_size=(320, 320),     # make concept refs bigger/clearer
    ref_col_scale=2.6,            # widen concept column
    concept_row_scale=1.7,       # taller concept rows
)

fig
# fig.savefig("/home/heydari/FHHI-XAI/output/pcx/pidnet_flood/layer5.0.conv1/panel_refs3_per_proto_with_matrix.png", dpi=300)

In [ ]:
# after plot_proto_concept_matrix(...)
vals = meta["values"]
print("min", vals.min(), "max", vals.max(), "mean", vals.mean())
print("largest 10:", np.sort(vals.flatten())[-10:])